In [2]:
import pandas as pd
import joblib
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Carregando e limpando os dados
df = pd.read_csv('../data/ai4i2020.csv')
df_ml = df.drop(['UDI', 'Product ID'], axis=1)
df_ml['Type'] = df_ml['Type'].map({'L': 0, 'M': 1, 'H': 2})

X = df_ml.drop(['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1)
y = df_ml['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- A MAGIA DO MLFLOW (VERSÃO À PROVA DE BUGS DO WINDOWS) ---
print("A configurar a Base de Dados do MLflow...")

# Usamos um ficheiro SQLite em vez de pastas complexas
mlflow.set_tracking_uri("sqlite:///mlflow.db")

nome_do_experimento = "Manutencao_Preditiva_API"

# Criamos o experimento forçando os modelos a ficarem numa pasta simples
try:
    mlflow.create_experiment(nome_do_experimento, artifact_location="./artefactos_mlflow")
except:
    pass # Ignora se o experimento já existir

mlflow.set_experiment(nome_do_experimento)

print("A treinar e a registar os dados...")
with mlflow.start_run():
    
    n_estimators = 100
    class_weight = 'balanced'

    modelo = RandomForestClassifier(n_estimators=n_estimators, class_weight=class_weight, random_state=42)
    modelo.fit(X_train, y_train)
    
    previsoes = modelo.predict(X_test)
    
    f1 = f1_score(y_test, previsoes, average='macro')
    acc = accuracy_score(y_test, previsoes)
    
    print("\n--- Relatório de Performance do Modelo ---")
    print(classification_report(y_test, previsoes))
    
    # Registando métricas na Base de Dados
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("class_weight", class_weight)
    mlflow.log_metric("f1_score_macro", f1)
    mlflow.log_metric("accuracy", acc)
    
    mlflow.sklearn.log_model(modelo, "modelo-random-forest")
    
    # Guardando o modelo localmente para a sua FastAPI usar
    os.makedirs('../models', exist_ok=True)
    joblib.dump(modelo, '../models/predictive_maintenance_model.joblib')
    print("\n✅ Sucesso: Modelo treinado e registado na Base de Dados do MLflow!")

2026/05/24 19:04:14 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/24 19:04:14 INFO mlflow.store.db.utils: Updating database tables


A configurar a Base de Dados do MLflow...
A treinar e a registar os dados...


2026/05/24 19:04:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/24 19:04:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



--- Relatório de Performance do Modelo ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.88      0.56      0.68        68

    accuracy                           0.98      2000
   macro avg       0.93      0.78      0.84      2000
weighted avg       0.98      0.98      0.98      2000


✅ Sucesso: Modelo treinado e registado na Base de Dados do MLflow!
